# LDM training orchestration

This notebook uses the private GitHub checkout at `/content/ldm`, downloads a pinned private Hugging Face dataset revision, and calls the package. Clone `https://github.com/appodoom/ldm.git` into `/content/ldm` with your authorized GitHub access before starting.


In [ ]:
from pathlib import Path
project = Path("/content/ldm")
assert (project / "pyproject.toml").exists(), "Clone the private appodoom/ldm repository into /content/ldm first"


In [ ]:
import os, subprocess, sys
print("Python:", sys.version.split()[0])
os.chdir(project)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Load the pinned private dataset

Sign in to Hugging Face in this Colab session. The normal and decorated files share names and cycle boundaries.


In [ ]:
from huggingface_hub import notebook_login, snapshot_download
notebook_login()
data_root = Path(snapshot_download(
    repo_id="Appodoom/ldm-data", repo_type="dataset",
    revision="34fb2449a25472e856a4db0aa85e280e9a82a8b2",
    allow_patterns="five-line/**", local_dir=project / ".derbake",
))
ordinary = data_root / "five-line" / "normal"
decorated = data_root / "five-line" / "decorated"
assert len(list(ordinary.glob("*.derbake"))) == 116
assert len(list(decorated.glob("*.derbake"))) == 116


## Prepare one variant

The ordinary and decorated paths must refer to the paired converted five-line corpus. Each file is split by complete cycles into approximately 70/20/10.


In [ ]:
variant = "with_subd"  # or "no_subd"
dataset = project / "datasets" / variant
subprocess.run([sys.executable, "-m", "ldm.cli", "prepare", "--ordinary", str(ordinary), "--decorated", str(decorated), "--variant", variant, "--output", str(dataset)], check=True)


## Train when approved

This is deliberately not run automatically. Use durable Drive paths for `run` and `dataset` in a real Colab session.


In [ ]:
run = project / "runs" / variant
# Uncomment when ready to train:
# subprocess.run([sys.executable, "-m", "ldm.cli", "train", "--dataset", str(dataset), "--run", str(run), "--epochs", "10"], check=True)


## Build a local HF snapshot after training

Validate the snapshot before any Hub upload.


In [ ]:
snapshot = project / "snapshots" / variant
# subprocess.run([sys.executable, "-m", "ldm.cli", "snapshot", "--run", str(run), "--output", str(snapshot)], check=True)
# subprocess.run([sys.executable, "-m", "ldm.cli", "validate", "--snapshot", str(snapshot)], check=True)
